# Detecting noisy monitors

This notebook shows how to detect noisy monitors in a dataset using the WhyLabs Monitor Diagnoser. It uses the diagnoser to automatically detect the noisiest monitor for dataset, get a diagnosis of
the conditions causing the noise, get recommended changes and where automatable, apply those changes.

## Install requirements

In [5]:
# %pip install whylabs-toolkit[diagnoser]

## Setup whylabs API connection

First, set up the information to connect to WhyLabs. Update the org_id, dataset_id and api_key in the following before running it.


In [6]:
import getpass
from whylabs_toolkit.monitor.diagnoser.helpers.utils import env_setup

org_id = input("Enter org ID")
dataset_id = input("Enter model/dataset ID")
api_key = getpass.getpass("Enter API key")
api_endpoint = 'https://api.whylabsapp.com'

env_setup(
    org_id=org_id,
    dataset_id=dataset_id,
    api_key=api_key,
    whylabs_endpoint=api_endpoint
)

Initialize the Monitor Diagnoser with the org_id and dataset_id.

In [7]:
from whylabs_toolkit.monitor.diagnoser.monitor_diagnoser import MonitorDiagnoser
diagnoser = MonitorDiagnoser(org_id, dataset_id)

## Run the default diagnosis

With no further input, the diagnoser will make a series of calls to identify the noisiest monitor, segment and columns; and then perform a diagnosis.

In [8]:
monitor_report = diagnoser.diagnose()
monitor_report

MonitorDiagnosisReport(orgId='org-0', datasetId='model-0', analyzerId='frequent-items-drift-analyzer-x2hr9z', interval='2024-04-02T00:00:00.000Z/2024-05-02T00:00:00.000Z', expectedBatchCount=0, diagnosticData=DiagnosticDataSummary(diagnosticSegment=Segment(tags=[]), diagnosticProfile=ProfileSummary(minRowName='desc', minRowCount=1674392, maxRowName='desc', maxRowCount=1674392), diagnosticBatches=BatchesSummary(minBatchName='desc', minBatchCount=30, maxBatchName='desc', maxBatchCount=30), analysisResults=AnalysisResultsSummary(results=ResultRecord(diagnosedColumnCount=26, batchCount=30), failures=FailureRecord(totalFailuresCount=0, maxFailuresCount=0, meanFailuresCount=0, byColumnCount=[], byTypeCount=[]), anomalies=AnomalyRecord(totalAnomalyCount=34, maxAnomalyCount=30, meanAnomalyCount=11, batchCount=30, byColumnCount=[NamedCount(name='issue_d', count=30), NamedCount(name='url', count=3), NamedCount(name='desc', count=1)], byColumnBatchCount=[NamedCount(name='addr_state', count=30), N

In [9]:
print(monitor_report.describe())

Diagnosis is for monitor "frequent-items-drift-monitor-x2hr9z" [frequent-items-drift-monitor-x2hr9z] in model-0 org-0, over interval 2024-04-02T00:00:00.000Z/2024-05-02T00:00:00.000Z.
Monitor has 1 notification actions ['email'].

Analyzer is drift configuration for frequent_items metric with TrailingWindow baseline.
Analyzer "frequent-items-drift-analyzer-x2hr9z" targets 27 columns and ran on 26 columns in the diagnosed segment.


Diagnostic segment is "overall".
Diagnostic interval contains 30 batches.

Diagnostic interval rollup contains 1674392 rows for the diagnosed columns.

Analysis results summary:
Found non-failed results for 26 columns and 30 batches.
Found 34 anomalies in 3 columns, with up to 100.0% (30) batches having anomalies per column and 36.7% (11.0) on average.
Columns with anomalies are:
|    | 0               |
|---:|:----------------|
|  0 | ('issue_d', 30) |
|  1 | ('url', 3)      |
|  2 | ('desc', 1)     |

No failures were detected.

Conditions that may impact 

The monitor report can be serialized to a JSON file for later use.

In [10]:
with open('monitor_report.json', 'w') as f:
    f.write(monitor_report.json())

In [11]:
from whylabs_toolkit.monitor.diagnoser.models import MonitorDiagnosisReport

with open('monitor_report.json', 'r') as f:
    monitor_report = MonitorDiagnosisReport.parse_raw(f.read())
print(monitor_report.json(indent=2))

{
  "orgId": "org-0",
  "datasetId": "model-0",
  "analyzerId": "frequent-items-drift-analyzer-x2hr9z",
  "interval": "2024-04-02T00:00:00.000Z/2024-05-02T00:00:00.000Z",
  "expectedBatchCount": 0,
  "diagnosticData": {
    "diagnosticSegment": {
      "tags": []
    },
    "diagnosticProfile": {
      "minRowName": "desc",
      "minRowCount": 1674392,
      "maxRowName": "desc",
      "maxRowCount": 1674392
    },
    "diagnosticBatches": {
      "minBatchName": "desc",
      "minBatchCount": 30,
      "maxBatchName": "desc",
      "maxBatchCount": 30
    },
    "analysisResults": {
      "results": {
        "diagnosedColumnCount": 26,
        "batchCount": 30
      },
      "failures": {
        "totalFailuresCount": 0,
        "maxFailuresCount": 0,
        "meanFailuresCount": 0,
        "byColumnCount": [],
        "byTypeCount": []
      },
      "anomalies": {
        "totalAnomalyCount": 34,
        "maxAnomalyCount": 30,
        "meanAnomalyCount": 11,
        "batchCount": 

## Ask for recommended changes

Given the diagnosis report for the monitor, the ChangeRecommender will recommend changes to make to the monitor. By default it will make recommendations for all columns where it has detected noise-related conditions. Set the `min_anomaly_count` property to restrict this to only columns that caused a certain number of anomalies.


In [12]:
from whylabs_toolkit.monitor.diagnoser.recommendation.change_recommender import ChangeRecommender

recommender = ChangeRecommender(monitor_report)
recommender.min_anomaly_count = 1
changes = recommender.recommend()
print('\n'.join([f'{i+1}. {c.describe()}' for i, c in enumerate(changes)]))

1. Remove columns from the analyzer for ['desc', 'issue_d', 'url']
2. Make a manual change to the analyzer to address small_nonnull_batches: less than 500 non-null records in 50% or more of the batches for ['desc']


## Execute automatable changes

A subset of recommended changes can be executed automatically by the recommender. Pass the ones you want to make into the `make_changes` call, or pass all changes if you want it to make all of the automatable changes.

In [13]:
automatable_changes = [c for c in changes if c.can_automate()]
print('\n'.join([c.describe() for c in automatable_changes]))

Remove columns from the analyzer for ['desc', 'issue_d', 'url']


In [14]:
change_results = recommender.make_changes(automatable_changes)
print(change_results.describe())

Successfully made the following changes:
	* Remove columns from the analyzer for ['desc', 'issue_d', 'url']


Note that the monitor will still appear to the diagnoser as the noisiest monitor until enough time has passed for the impact of the monitor changes to be observed. You may want to use the WhyLabs preview UI to view what impacts may be expected from the change.

## Reviewing other noisy monitors

The diagnoser can be used to review other noisy monitors in the dataset. The `noisy_monitors` property will return a list of the noisiest monitors, and the `monitor_id_to_diagnose` property can be set to the monitor_id of the monitor to diagnose.

In [15]:
import pandas as pd
noisy_monitors_df = pd.DataFrame.from_records([m.dict() for m in diagnoser.noisy_monitors])
noisy_monitors_df

,monitor_id,analyzer_id,metric,column_count,segment_count,anomaly_count,max_anomaly_per_column,min_anomaly_per_column,avg_anomaly_per_column,action_count,action_targets
0,frequent-items-drift-monitor-x2hr9z,frequent-items-drift-analyzer-x2hr9z,frequent_items,3,1,34,30,1,11,1,[email]
1,discrete-distribution-22ef37c9-monitor,discrete-distribution-22ef37c9,frequent_items,3,1,34,30,1,11,0,[]
2,smoggy-chartreuse-owl-3387,smoggy-chartreuse-owl-3387-analyzer,frequent_items,3,1,34,30,1,11,0,[]
3,frequent-items-drift-monitor-bx6m80,frequent-items-drift-analyzer-bx6m80,frequent_items,3,1,34,30,1,11,0,[]
4,frequent-items-drift-monitor-mat0jo,frequent-items-drift-analyzer-mat0jo,frequent_items,3,1,34,30,1,11,2,"[email, slack]"
5,frequent-items-drift-monitor-01rbfl,frequent-items-drift-analyzer-01rbfl,frequent_items,3,1,34,30,1,11,1,[email]
6,frequent-items-drift-monitor-0foigt,frequent-items-drift-analyzer-0foigt,frequent_items,3,1,34,30,1,11,0,[]
7,frequent-items-drift-monitor-3c0hc2,frequent-items-drift-analyzer-3c0hc2,frequent_items,3,1,34,30,1,11,1,[email]
8,frequent-items-drift-monitor-9gmtix,frequent-items-drift-analyzer-9gmtix,frequent_items,3,1,34,30,1,11,1,[email]
9,elated-palegreen-jaguar-6432,elated-palegreen-jaguar-6432-analyzer,histogram,9,1,75,19,2,8,0,[]


In [16]:
diagnoser.monitor_id_to_diagnose = noisy_monitors_df.iloc[1]['monitor_id']
print(diagnoser.monitor_id_to_diagnose)
monitor_report = diagnoser.diagnose()
print(monitor_report.describe())

discrete-distribution-22ef37c9-monitor
Diagnosis is for monitor "discrete-distribution-22ef37c9-monitor" [discrete-distribution-22ef37c9-monitor] in model-0 org-0, over interval 2024-04-02T00:00:00.000Z/2024-05-02T00:00:00.000Z.

Analyzer is drift configuration for frequent_items metric with TrailingWindow baseline.
Analyzer "discrete-distribution-22ef37c9" targets 30 columns and ran on 26 columns in the diagnosed segment.


Diagnostic segment is "overall".
Diagnostic interval contains 30 batches.

Diagnostic interval rollup contains 1674392 rows for the diagnosed columns.

Analysis results summary:
Found non-failed results for 26 columns and 30 batches.
Found 34 anomalies in 3 columns, with up to 100.0% (30) batches having anomalies per column and 36.7% (11.0) on average.
Columns with anomalies are:
|    | 0               |
|---:|:----------------|
|  0 | ('issue_d', 30) |
|  1 | ('url', 3)      |
|  2 | ('desc', 1)     |

No failures were detected.

Conditions that may impact diagnos

You can also use the `noisy_monitors_with_actions` property to prioritize noise in monitors with actions, as these are most likely to cause alert fatigue.

In [17]:
pd.DataFrame.from_records([m.dict() for m in diagnoser.noisy_monitors_with_actions])


,monitor_id,analyzer_id,metric,column_count,segment_count,anomaly_count,max_anomaly_per_column,min_anomaly_per_column,avg_anomaly_per_column,action_count,action_targets
0,frequent-items-drift-monitor-x2hr9z,frequent-items-drift-analyzer-x2hr9z,frequent_items,3,1,34,30,1,11,1,[email]
1,frequent-items-drift-monitor-mat0jo,frequent-items-drift-analyzer-mat0jo,frequent_items,3,1,34,30,1,11,2,"[email, slack]"
2,frequent-items-drift-monitor-01rbfl,frequent-items-drift-analyzer-01rbfl,frequent_items,3,1,34,30,1,11,1,[email]
3,frequent-items-drift-monitor-3c0hc2,frequent-items-drift-analyzer-3c0hc2,frequent_items,3,1,34,30,1,11,1,[email]
4,frequent-items-drift-monitor-9gmtix,frequent-items-drift-analyzer-9gmtix,frequent_items,3,1,34,30,1,11,1,[email]
5,inferred-data-type-fec5a735-monitor,inferred-data-type-fec5a735,inferred_data_type,1,1,14,14,14,14,2,"[email, slack]"
